# Vidhi llama.cpp model comparison

This notebook evaluates prawinin/vidhi using the repository's GGUF model through llama.cpp. It uses the same 12 prompt instances and output metadata format as the Transformers notebooks.

The default run processes one prompt instance. Change RUN_LIMIT to None after the smoke test to run all 12 instances.

## 1. Install dependencies

Vidhi is distributed for llama.cpp-style inference rather than the normal Transformers loading path.

In [ ]:
!pip -q install huggingface_hub llama-cpp-python

## 2. Imports, optional Hugging Face login, and cache locations

In [ ]:
import gc
import json
os
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

from huggingface_hub import hf_hub_download, list_repo_files, login
from llama_cpp import Llama

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_CACHE = Path("/content/hf_cache")
OUTPUT_ROOT = Path("/content/experiment_outputs/model_comparison")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Vidhi cache:", MODEL_CACHE)

## 3. Upload and load the prompt configuration

In [ ]:
from google.colab import files

CONFIG_PATH = Path("/content/prompt_configurations.json")
if not CONFIG_PATH.exists():
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    if not uploaded_names:
        raise FileNotFoundError("Upload prompt_configurations.json before continuing.")
    CONFIG_PATH = Path("/content") / uploaded_names[0]

PROMPT_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
PROMPT_INSTANCES = PROMPT_CONFIG["prompt_instances"]
assert len(PROMPT_INSTANCES) == 12, "Expected 12 prompt instances."
print("Loaded:", CONFIG_PATH)
print("Prompt instances:", len(PROMPT_INSTANCES))

In [ ]:
# ---------- 4-bit, and GPU setup ----------
%pip install -q bitsandbytes
import gc
import torch

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

clear_gpu_cache()

## 4. Select run size and llama.cpp settings

In [ ]:
MODEL_KEY = "vidhi"
MODEL_ID = "prawinin/vidhi"
RUN_LIMIT = 1                 # Set to None to process all 12 instances.
PROMPT_INSTANCE_IDS = None    # Or provide a list of IDs for a custom subset.
MAX_NEW_TOKENS = 1024
N_CTX = 8192
N_GPU_LAYERS = -1             # Falls back to CPU if the GPU backend is unavailable.
RUN_EXPERIMENT = True

## 5. Build prompts from the JSON configuration

In [ ]:
def render_allowed_labels(labels):
    lines = []
    for item in labels:
        if isinstance(item, dict):
            lines.append(f"- {item['label']}: {item['description']}")
        else:
            lines.append(f"- {item}")
    return "\n".join(lines)


def render_schema(schema):
    if isinstance(schema, dict):
        return json.dumps(schema, ensure_ascii=False)
    return str(schema)


def build_prompt(instance):
    prompt_set = PROMPT_CONFIG["prompt_sets"][instance["prompt_set"]]
    return (
        f"{prompt_set['instruction']}\n\n"
        f"Allowed labels:\n{render_allowed_labels(prompt_set['allowed_labels'])}\n\n"
        f"Schema:\n{render_schema(prompt_set['schema'])}\n\n"
        f"DOCUMENT:\n{instance['document']['text']}\n\n"
        "OUTPUT:"
    )


def select_instances():
    instances = PROMPT_INSTANCES
    if PROMPT_INSTANCE_IDS is not None:
        wanted = set(PROMPT_INSTANCE_IDS)
        instances = [item for item in instances if item["id"] in wanted]
    if RUN_LIMIT is not None:
        instances = instances[:RUN_LIMIT]
    if not instances:
        raise ValueError("No prompt instances selected.")
    return instances


print(build_prompt(select_instances()[0])[:2000])

## 6. Download the GGUF checkpoint, generate, and save raw outputs

In [ ]:
def find_gguf_file():
    files_in_repo = list_repo_files(MODEL_ID, token=HF_TOKEN or None)
    candidates = [name for name in files_in_repo if name.lower().endswith(".gguf")]
    if not candidates:
        raise FileNotFoundError("No GGUF file was found in the Vidhi repository.")
    preferred = [name for name in candidates if "q4_k_m" in name.lower()]
    return preferred[0] if preferred else candidates[0]


def load_vidhi():
    gguf_name = find_gguf_file()
    model_path = hf_hub_download(
        repo_id=MODEL_ID,
        filename=gguf_name,
        cache_dir=str(MODEL_CACHE / MODEL_KEY),
        token=HF_TOKEN or None,
    )
    try:
        llm = Llama(
            model_path=model_path,
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            verbose=False,
        )
        backend = "llama_cpp_gpu_requested"
    except Exception as first_error:
        print("GPU llama.cpp loading failed; retrying on CPU:", first_error)
        llm = Llama(model_path=model_path, n_ctx=N_CTX, n_gpu_layers=0, verbose=False)
        backend = "llama_cpp_cpu_fallback"
    return llm, gguf_name, backend


def generate_one(llm, prompt):
    started = time.perf_counter()
    response = llm(
        prompt,
        max_tokens=MAX_NEW_TOKENS,
        temperature=0.0,
        top_p=1.0,
        echo=False,
    )
    raw_output = response["choices"][0]["text"]
    return raw_output, time.perf_counter() - started


def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def run_experiment():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    output_path = OUTPUT_ROOT / MODEL_KEY / f"{run_id}.json"
    selected = select_instances()
    payload = {
        "status": "running",
        "run_id": run_id,
        "model_key": MODEL_KEY,
        "model_id": MODEL_ID,
        "loading_mode": "llama_cpp_gguf",
        "prompt_config_path": str(CONFIG_PATH),
        "selected_instance_ids": [item["id"] for item in selected],
        "run_limit": RUN_LIMIT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "n_ctx": N_CTX,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": [],
    }
    save_json(output_path, payload)
    llm = None
    try:
        llm, gguf_name, backend = load_vidhi()
        payload["gguf_file"] = gguf_name
        payload["backend"] = backend
        for instance in selected:
            prompt = build_prompt(instance)
            record = {
                "instance_id": instance["id"],
                "prompt_set": instance["prompt_set"],
                "source_type": instance["document"]["source_type"],
                "document_word_count": instance["document"]["actual_word_count"],
                "prompt": prompt,
                "raw_output": None,
                "prompt_mode": "plain_text_llama_cpp",
                "seconds": None,
                "status": "error",
                "error": None,
            }
            try:
                raw, seconds = generate_one(llm, prompt)
                record.update(raw_output=raw, seconds=seconds, status="completed")
            except Exception as exc:
                record["error"] = f"{type(exc).__name__}: {exc}"
            payload["records"].append(record)
            save_json(output_path, payload)
    except Exception as exc:
        payload["load_error"] = f"{type(exc).__name__}: {exc}"
    finally:
        if llm is not None:
            del llm
        gc.collect()
    payload["status"] = "completed" if "load_error" not in payload else "load_error"
    save_json(output_path, payload)
    print(f"Saved: {output_path}")
    print("Completed records:", sum(r["status"] == "completed" for r in payload["records"]))
    return payload


if RUN_EXPERIMENT:
    result = run_experiment()